# Paper 1 — Dynamic Graph-Temporal Transformer for Multi-Product Food Price Forecasting

**Research question.** Do relationships among food products — estimated statistically, or learned and allowed
to change over time — improve multi-horizon price forecasts compared with univariate and conventional
multivariate models?

**Falsifiable claim tested here.** *How the product graph is constructed explains more variance in forecast
accuracy than the choice of graph architecture.* We test it with a full factorial experiment
(10 graph constructions x 3 architectures x seeds x rolling folds) and a variance decomposition.

**Contributions**

1. A leak-free graph-construction protocol for sticky, web-scraped e-commerce prices that removes
   scraper-outage days, gap-end days and platform batch-update days before estimating co-movement.
2. DGTT, a Dynamic Graph-Temporal Transformer: a patch-based temporal encoder, a graph learner that combines a
   statistical prior, learned static node embeddings and time-varying query-key affinities (top-k sparsified),
   and cross-product attention over the resulting neighbourhoods.
3. A factorial graph x architecture study with eta-squared decomposition, an ablation of every DGTT component,
   and heterogeneity analyses by volatility, activity tier and category, all with Diebold-Mariano, Wilcoxon and
   bootstrap inference.

**Runtime (Colab T4, rough).** Data prep < 1 min; statistical baselines 15-30 min; LightGBM and deep
baselines ~30 min; the factorial (10 graphs x 3 architectures x 2 seeds x 3 folds = 180 runs) and the ablations
are the bulk — expect several hours in total. Every run checkpoints to Drive after each epoch and finished runs
are skipped, so the notebook can be re-run across several Colab sessions. Reduce `FACT_SEEDS`, `SEEDS` or
`FOLD_IDS` in section 3 for a faster first pass.

**Outputs** (all on Drive under `food_price_research/paper1_graph_temporal/`): `figures/*.pdf|png`,
`tables/*.tex|csv`, `results/` (predictions and run registry), `checkpoints/`.

## 2. Dataset description

In [ ]:
# ==== 0. CONFIGURATION & GOOGLE DRIVE =====================================
PAPER = "paper1_graph_temporal"
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("FPR_SMOKE", "0") == "1"   # True -> tiny, fast end-to-end test run
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/food_price_research"
else:
    ROOT = os.environ.get("FPR_ROOT", os.path.abspath("food_price_research"))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'statsforecast', 'lightgbm', 'statsmodels'], check=False)

from pathlib import Path
ROOT = Path(ROOT); RAW = ROOT / "raw"; RAW.mkdir(parents=True, exist_ok=True)
PREP_DIR = ROOT / "shared_prep"                      # cleaned panel, shared by all five papers
PAPER_DIR = ROOT / PAPER
FIG_DIR, TAB_DIR, RES_DIR, CKPT_DIR = [PAPER_DIR / s for s in ("figures", "tables", "results", "checkpoints")]
for d in (PREP_DIR, FIG_DIR, TAB_DIR, RES_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

cands = sorted(RAW.glob("[dD]ataset-2024.*"))
if not cands and IN_COLAB:
    from google.colab import files
    print("Upload dataset-2024.csv (or Dataset-2024.xlsx) - it is saved to", RAW)
    for k, v in files.upload().items():
        (RAW / k).write_bytes(v)
    cands = sorted(RAW.glob("[dD]ataset-2024.*"))
assert cands, f"Put dataset-2024.csv in {RAW}"
DATA_PATH = cands[0]
print("paper:", PAPER, "| data:", DATA_PATH, "| smoke:", SMOKE)

## 1. Shared data preparation (identical in all five notebooks)

The raw panel is a wide table (`Name | 2024-01-02 | ... | 2025-02-05`): 5,976 SKUs x 401 days, already
complete. Quality control, all estimated on training data only:

* **Scraper outages.** On 56 days not a single SKU changes price (the median day has ~80 changes): the scraper
  did not run and prices were carried forward. These cells are *imputed*, so they are excluded from every loss,
  metric target and graph statistic. The first day after an outage (a *gap-end* day) accumulates several days of
  changes and is excluded from co-movement statistics.
* **Spike-and-revert repair.** A jump of more than x1.5 that returns to within 5 % of the previous level within
  7 days is treated as a scrape error (wrong variant shown) and replaced by the previous level.
* **Variant breaks.** A persistent level shift of more than x3 within the training window is almost surely a
  re-used product slot; those SKUs are excluded.
* **Platform batch-update days.** Days whose number of simultaneous changes exceeds median + 5 MAD. Batch
  updates create spurious co-movement, so they are excluded from correlation-type statistics.
* **Category tagger.** Priority-ordered keyword rules on product names; non-food SKUs are kept as a control
  group and excluded from headline results.
* **Activity tiers** (on the earliest training window, so no look-ahead): T0 food SKUs that never change,
  T1 1-9 changes, T2 >= 10 changes. Headline results use the *active* universe T1 u T2.
* **Rolling-origin evaluation.** Three folds, each with a 30-day validation and a 30-day test window
  (test windows: 2024-11-08..12-07, 12-08..01-06, 2025-01-07..02-05). Models never see validation or test
  targets during fitting; forecasts at each origin use only data up to that origin.
* **Targets.** 100 x log(P[t+h] / P[t]) for h = 1..30; metrics are computed in price space at h in {1,3,7,14,30}.
  Because most prices are sticky we also report **CC-sMAPE** (sMAPE only where the price actually changes) and
  **change AUPRC**, alongside MAE, RMSE, MAPE, sMAPE, WAPE and MASE.

In [ ]:
# ============================================================================
# SECTION 1 — SHARED DATA PREPARATION (identical, deterministic in every notebook)
# ============================================================================
import re, json, time, math, pickle, hashlib, warnings, itertools
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1.1  Category tagger (rule-based, from product names; priority-ordered)
# ---------------------------------------------------------------------------
# Non-food rules are checked FIRST so that e.g. "hair oil" never becomes edible
# oil and "baby shampoo" never becomes baby food.
CATEGORY_RULES = [
    # ---- non-food -------------------------------------------------------
    ("NF: Personal Care", r"shampoo|conditioner|face ?wash|facewash|facial|soap|lotion|deodorant|deo |roll on|perfume|body spray|"
                          r"toothpaste|toothbrush|tooth ?brush|mouthwash|hair oil|hair color|hair colour|hair removal|hair treatment|"
                          r"serum|sunscreen|sun protect|talcum|cream(?!.*(biscuit|wafer|cake|cheese|ice cream|milk|roll|sandwich))|"
                          r"vaseline|petroleum jelly|nose strip|shaving|razor|blade|sanitary|napkin|pad[s]? |tampon|condom|lip ?balm|"
                          r"lipstick|nail|kajal|mascara|glycerine|cleansing|scrub|makeup|cosmetic|beauty|skin|aloe ?vera gel|"
                          r"hand ?wash|body wash|shower gel|cotton bud|wipes|essential oil|facial oil|massage oil|coconut oil hair|"
                          r"boroplus|moisturi|toner|face mask|mask|blackhead|dandruff|gatsby|gloss|hair gel|lip colou?r|hair|shave|sanitizer|baby wash|baby bath|\blip"),
    ("NF: Baby Care",     r"diaper|nappy|pant [smlx]|belt \d|feeding bottle|bottles cleanser|nipple|baby pillow|nima|teether|"
                          r"baby lotion|baby oil|baby soap|baby shampoo|baby powder|baby wipes|pacifier|romper|cot sheet|pants .*kg"),
    ("NF: Household",     r"clean(er|ing)|detergent|dish ?wash|washing|bleach|disinfect|toilet|harpic|lizol|air freshener|"
                          r"freshener|aerosol|mosquito|insect|coil|killer|repellent|tissue|napkin|foil|cling film|garbage|"
                          r"broom|mop|brush|sponge|scrubber|battery|bulb|light|candle|matchbox|match box|lighter|polish|"
                          r"glass cleaner|floor|mat |mat$|pillow|towel|bucket|container|organizer|box$|jar |bottle(?!.*(juice|drink|water|milk|sauce|oil))|"
                          r"plate|bowl|spoon|glass |mug|cup(?! noodles)|tray|chopping board|knife|rolling pin|leveler|grill|net$|pan |pot |kettle|"
                          r"flask|tiffin|lunch box|hanger|umbrella|finish household|powerball|fabric|softener|surf excel|rin |wheel |"
                          r"cat food|dog food|pet |\bcat\b|kitten|\bdog\b|smartheart|pro diet|whiskas|pedigree|kitty|wrap|mixer|apron|strainer|duster|dish cover|holder|litter|"
                          r"spray|cockroach|\bhit\b|diesel|\bmobil\b|lubricant|\bcover\b|sheet"),
    ("NF: Health",        r"tablet|capsule|syrup(?!.*(chocolate|strawberry))|vitamin|antiseptic|dettol liquid|savlon|bandage|"
                          r"plaster|thermometer|oral saline|ors|pain relief|balm|inhaler|medicine|n95|surgical|glucose(?! biscuit)|"
                          r"test kit|pregnancy"),
    ("NF: Stationery & Misc", r"pencil|pen |pen$|ball ?pen|marker|highlighter|eraser|sharpener|notebook|note ?book|file |folder|"
                          r"stapler|punch|scissor|cutter|glue|tape|calculator|compass|geometry|ruler|a4|paper|envelope|sticker|"
                          r"flag|toy|puzzle|ball |badminton|shuttlecock|cricket|football|game|doll|remote|charger|cable|"
                          r"earphone|headphone|usb|mobile|phone|fusion set|moistener|clip|board|diary|book|colour pencil|crayon|"
                          r"gift|decoration|balloon|travel|bag|wallet|\bcard\b|\bbear\b|\bcar\b|instrument|acrylic|glitter|sash|tasbih|"
                          r"\bbat\b|lace|drawing|flower"),
    # ---- food ---------------------------------------------------------------
    ("Baby Food",         r"formula|infant|baby food|cerelac|nan optipro|lactogen|biomil|growing up milk|\(0-6|\(6-12|\(1-3|"
                          r"nido 1|nido 3|cereal.*(tin|baby)|mother's smile"),
    ("Beverages",         r"juice|drink|beverage|electrolyte|soda|cola|7 ?up|sprite|fanta|pepsi|coca|mojo|speed|energy|water|lassi|milk shake|"
                          r"shake|tang|horlicks|boost|ovaltine|bournvita|malt|squash|syrup|can$|smoothie|mocktail|aloe drink|frooto|frutika"),
    ("Tea & Coffee",      r"\btea\b|tea bag|green tea|coffee|nescafe|infusion|chai"),
    ("Dairy & Eggs",      r"milk|cheese|butter(?!.*(cookie|biscuit|peanut))|yogurt|yoghurt|\bdoi\b|curd|cream cheese|paneer|"
                          r"ghee|egg|dim\b|dairy|uht|evaporated|condensed|ice cream|kulfi"),
    ("Edible Oil",        r"soybean oil|soyabean|mustard oil|sunflower oil|palm oil|rice bran|olive oil|canola|"
                          r"vegetable oil|cooking oil|sesame oil|\boil\b"),
    ("Fish & Seafood",    r"fish|ilish|hilsha|rui|rohu|katla|pangas|tilapia|shing|magur|koi|pabda|chingri|shrimp|prawn|"
                          r"crab|squid|baila|boal|shol|taki|poa|rupchanda|pomfret|loitta|kachki|mola|tengra|shutki|dried fish"),
    ("Meat & Poultry",    r"beef|mutton|goat|lamb|chicken|broiler|deshi murgi|murgi|duck(?! eggs)|pigeon|quail|meat|"
                          r"liver|gizzard|keema|mince|bone in|boneless|drumstick|breast|wings|sausage|salami|nugget|patty|kebab"),
    ("Frozen & Ready",     r"frozen|ready to cook|ready mix|paratha|puri|samosa|singara|spring roll|haleem mix|biryani mix|"
                          r"instant|french fries|burger|pizza(?! sauce)"),
    ("Rice & Grains",     r"\brice\b|chinigura|basmati|miniket|nazirshail|atop|kalijira|polao|poha|chira|muri|khoi|oats|"
                          r"cornflakes|corn flakes|cereal|muesli|barley|quinoa|semai|shemai|laccha|lachcha|vermicelli|wheat"),
    ("Flour & Bakery",    r"atta|maida|flour|suji|semolina|besan|bread|bun|cake|rusk|toast|bakorkhani|bakery|dough|yeast|"
                          r"baking|pastry|croissant|donut|ruti|chapati|roti"),
    ("Lentils & Pulses",  r"\bdal\b|lentil|masur|moong|mung|mashkalai|chola|chick ?pea|kabuli|anchor dal|khesari|"
                          r"motor|peas|beans?\b|rajma"),
    ("Spices & Condiments", r"masala|spice|powder|chili|chilli|turmeric|holud|coriander|cumin|zeera|jeera|cardamom|elach|"
                          r"cinnamon|daruchini|clove|lobongo|black pepper|golmorich|bay leaf|tejpata|nutmeg|jayfal|mace|"
                          r"fenugreek|methi|panch phoron|mix$|salt|vinegar|flakes|seasoning|stock|soup|bouillon|gelatine|"
                          r"food colou?r|essence|baking soda|kewra|rose water|seed|herbs|china grass|mix\s*$"),
    ("Sugar & Sweeteners", r"sugar|chini|\bgur\b|jaggery|honey|mishri|sweetener|candy|chocolate|toffee|lollipop|jelly|"
                          r"marshmallow|gum|mint"),
    ("Snacks & Biscuits", r"biscuit|cookie|cracker|wafer|chips|chanachur|namkeen|dalmoth|bhujia|nuts|peanut(?! butter)|"
                          r"cashew|kaju|almond|pista|raisin|kismis|dates|khejur|popcorn|puffs|roll$|snack|bar |bar$|"
                          r"chanachur|jhal muri|sticks|crisps|pringles|kurkure|lays|badam"),
    ("Sauces & Spreads",  r"sauce|ketchup|mayonnaise|mayo|\bjam\b|jelly|spread|pickle|achar|chutney|peanut butter|nutella|"
                          r"dressing|paste|puree|olive(?! oil)"),
    ("Noodles & Pasta",   r"noodle|ramen|pasta|macaroni|spaghetti|fusilli|penne|maggi|cup noodles"),
    ("Fruits",            r"mango|aam|banana|kola\b|apple|orange|malta|grape|guava|peyara|papaya|pape|pineapple|anaros|"
                          r"watermelon|tormuj|lemon|lebu|lime|pomegranate|anar|litchi|lichu|jackfruit|kathal|coconut|"
                          r"narikel|dates|berry|kiwi|pear|plum|prune|wood apple|kodbel|amra|jambura|sofeda|fruit"),
    ("Vegetables",        r"potato|alu|aloo|onion|peyaj|piyaj|garlic|roshun|ginger|ada\b|tomato|brinjal|begun|cucumber|"
                          r"shosha|pumpkin|kumra|gourd|lau|korola|potol|dherosh|okra|cabbage|cauliflower|kopi|carrot|gajor|"
                          r"radish|mula|spinach|shak|sak\b|lettuce|capsicum|green chili|kacha morich|beans|borboti|"
                          r"mushroom|coriander leaves|dhonia pata|mint leaves|pudina|vegetable|sobji|corn|bhutta|"
                          r"betel|shupari|paan|chal kumra|kochu|arum|shim|motorshuti|lemongrass|drumstick|broccoli|\bpui\b|microgreen|sheem"),
]
NONFOOD_PREFIX = "NF: "
_COMPILED = [(c, re.compile(p, re.I)) for c, p in CATEGORY_RULES]

def fix_mojibake(s: str) -> str:
    """Repair UTF-8 read as cp1252 (e.g. 'Â±50 gm' -> '±50 gm')."""
    try:
        t = s.encode("cp1252").decode("utf-8")
        return t
    except Exception:
        return s.replace("Â±", "±").replace("Â ", " ").replace("â€™", "'")

def tag_category(name: str) -> str:
    n = " " + name.lower().strip() + " "
    for cat, rx in _COMPILED:
        if rx.search(n):
            return cat
    return "Unclassified"

# ---------------------------------------------------------------------------
# 1.2  Load the wide panel  (Name | date1 | date2 | ...)
# ---------------------------------------------------------------------------
def load_panel(path):
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xls"):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path, encoding="utf-8-sig")
    df = df.rename(columns={df.columns[0]: "Name"})
    date_cols = [c for c in df.columns[1:]]
    dates = pd.to_datetime([str(c)[:10] for c in date_cols])
    P = df[date_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    names = df["Name"].astype(str).map(fix_mojibake).str.strip().tolist()
    return names, dates, P

# ---------------------------------------------------------------------------
# 1.3  Data-quality: interpolated-window mask, spike-and-revert repair,
#      persistent variant breaks, platform batch-update days
# ---------------------------------------------------------------------------
def detect_carried_forward_days(P, max_changes=2):
    """Scraper-outage days. When the scraper did not run, the panel carries the
    previous day's prices forward, so NOT A SINGLE product changes price that
    day (the median day has ~80 changes). These cells are imputed, not observed:
    they are excluded from every loss/metric target and from graph statistics.
    The first observed day after an outage accumulates several days of changes
    (a 'gap-end' day) and is flagged separately."""
    ch = np.concatenate([[10**9], (np.diff(P, axis=1) != 0).sum(axis=0)])
    cf = ch <= max_changes
    gap_end = np.zeros_like(cf)
    gap_end[1:] = cf[:-1] & ~cf[1:]
    return cf, gap_end

def repair_spikes(P, tau=np.log(1.5), max_len=7, tol=0.05):
    """Spike-and-revert repair: a jump |dlog| > tau that returns to within `tol`
    of the pre-jump level within `max_len` days is treated as a scrape error
    (wrong variant / pack size shown for a few days) and replaced by the
    pre-jump level. Returns cleaned matrix + per-series repair counts."""
    L = np.log(P)
    Q = P.copy(); n_rep = np.zeros(P.shape[0], dtype=int)
    T = P.shape[1]
    for i in range(P.shape[0]):
        x = L[i]
        d = np.diff(x)
        cand = np.where(np.abs(d) > tau)[0]
        t_skip = -1
        for j in cand:
            if j <= t_skip: continue
            base = x[j]
            for k in range(j + 1, min(j + 1 + max_len, T - 1)):
                if abs(x[k + 1] - base) <= tol:
                    Q[i, j + 1: k + 1] = P[i, j]
                    n_rep[i] += 1; t_skip = k
                    break
    return Q, n_rep

def detect_variant_breaks(P, t_end, thr=np.log(3.0)):
    """Persistent level shifts > x3 inside the training window: almost surely a
    re-used product slot (different pack/variant), not a price change."""
    d = np.abs(np.diff(np.log(P[:, : t_end + 1]), axis=1))
    return (d > thr).any(axis=1)

def detect_batch_days(P, t_end, k_mad=5.0):
    """Days where the number of simultaneous price changes is an outlier
    (median + k*MAD, threshold estimated on training days only)."""
    ch = (np.diff(P, axis=1) != 0).sum(axis=0)          # changes on day t (t>=1)
    ch = np.concatenate([[0], ch])
    tr = ch[1: t_end + 1]
    med = np.median(tr); mad = np.median(np.abs(tr - med)) + 1e-9
    thr = med + k_mad * 1.4826 * mad
    return ch > thr, ch, thr

# ---------------------------------------------------------------------------
# 1.4  Rolling-origin splits (3 folds, 30-day val + 30-day test each)
# ---------------------------------------------------------------------------
def make_folds(T, n_folds=3, test_len=30, val_len=30):
    folds = []
    for k in range(n_folds):
        test_end = T - 1 - test_len * (n_folds - 1 - k)
        test_start = test_end - test_len + 1
        val_end = test_start - 1; val_start = val_end - val_len + 1
        folds.append(dict(fold=k, train_end=val_start - 1, val_start=val_start, val_end=val_end,
                          test_start=test_start, test_end=test_end))
    return folds

# ---------------------------------------------------------------------------
# 1.5  Master prep (cached to Drive; identical in all five notebooks)
# ---------------------------------------------------------------------------
PREP_VERSION = "v1.1"

def build_prep(data_path, cache_dir, clean_spikes=True, force=False):
    cache_dir = Path(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)
    f = cache_dir / f"prep_{PREP_VERSION}.pkl"
    if f.exists() and not force:
        with open(f, "rb") as fh:
            prep = pickle.load(fh)
        print(f"[prep] loaded cached panel from {f}")
        return prep
    t0 = time.time()
    names, dates, P_raw = load_panel(data_path)
    N, T = P_raw.shape
    assert np.isfinite(P_raw).all() and (P_raw > 0).all(), "panel must be complete & positive"
    folds = make_folds(T)
    tier_end = folds[0]["train_end"]                    # leak-free: earliest training window
    cf_day, gap_end = detect_carried_forward_days(P_raw)
    P_clean, n_rep = repair_spikes(P_raw) if clean_spikes else (P_raw.copy(), np.zeros(N, int))
    variant_break = detect_variant_breaks(P_clean, tier_end)
    batch_day, daily_changes, batch_thr = detect_batch_days(P_clean, tier_end)
    cats = [tag_category(n) for n in names]
    is_food = np.array([not c.startswith(NONFOOD_PREFIX) and c != "Unclassified" for c in cats])
    # activity statistics on the tier window only (no look-ahead)
    dlog_tr = np.diff(np.log(P_clean[:, : tier_end + 1]), axis=1)
    n_changes = (dlog_tr != 0).sum(axis=1)
    vol = dlog_tr.std(axis=1)
    usable = is_food & ~variant_break
    tier = np.full(N, "T0", dtype=object)
    tier[usable & (n_changes >= 1)] = "T1"
    tier[usable & (n_changes >= 10)] = "T2"
    # volatility terciles among T1 (training-window std of log-returns)
    vol_bucket = np.full(N, "none", dtype=object)
    m = usable & (n_changes >= 1)
    q1, q2 = np.quantile(vol[m], [1 / 3, 2 / 3])
    vol_bucket[m & (vol <= q1)] = "low"
    vol_bucket[m & (vol > q1) & (vol <= q2)] = "medium"
    vol_bucket[m & (vol > q2)] = "high"
    meta = pd.DataFrame(dict(name=names, category=cats, is_food=is_food, variant_break=variant_break,
                             n_changes_train=n_changes, vol_train=vol, tier=tier, vol_bucket=vol_bucket,
                             spike_repairs=n_rep, mean_price_train=P_clean[:, : tier_end + 1].mean(axis=1)))
    prep = dict(names=names, dates=dates, P_raw=P_raw, P=P_clean, logP=np.log(P_clean), meta=meta,
                folds=folds, tier_end=tier_end, cf_day=cf_day, gap_end=gap_end,
                batch_day=batch_day, daily_changes=daily_changes, batch_thr=batch_thr,
                version=PREP_VERSION, built=time.strftime("%Y-%m-%d %H:%M"))
    with open(f, "wb") as fh:
        pickle.dump(prep, fh)
    print(f"[prep] built in {time.time()-t0:.1f}s and cached to {f}")
    return prep

def subset_idx(prep, tiers=("T1", "T2"), food_only=True, smoke_n=None, seed=0):
    meta = prep["meta"]
    m = meta.tier.isin(tiers).to_numpy()
    if food_only: m &= meta.is_food.to_numpy()
    idx = np.where(m)[0]
    if smoke_n is not None and len(idx) > smoke_n:
        rng = np.random.default_rng(seed); idx = np.sort(rng.choice(idx, smoke_n, replace=False))
    return idx

In [ ]:
# ============================================================================
# SECTION 1b — SHARED UTILITIES: figures, tables, checkpoints, windows, metrics
# ============================================================================
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats as sps

# ---------------------------------------------------------------------------
# Figure style (journal): 8-pt sans, thin marks, recessive grid, fixed palette
# ---------------------------------------------------------------------------
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GREY = "#8a8985"
COL1, COL2 = 3.5, 7.2                       # single / double column widths (inches)
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.6, "axes.edgecolor": "#52514e", "axes.grid": True, "grid.color": "#e4e3df",
    "grid.linewidth": 0.5, "lines.linewidth": 1.4, "lines.markersize": 4, "legend.frameon": False,
    "axes.prop_cycle": matplotlib.cycler(color=PALETTE), "pdf.fonttype": 42, "ps.fonttype": 42,
})

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{name}.{ext}")
    print(f"[fig] {name} -> {FIG_DIR}")
    plt.show(); plt.close(fig)

# ---------------------------------------------------------------------------
# Tables: CSV + LaTeX (booktabs), no jinja2 dependency; optional bolding of best
# ---------------------------------------------------------------------------
def _fmt(v, nd):
    if isinstance(v, (float, np.floating)):
        if not np.isfinite(v): return "--"
        return f"{v:.{nd}f}"
    return str(v).replace("_", r"\_").replace("&", r"\&").replace("%", r"\%")

def save_table(df, name, caption="", label=None, nd=3, bold_min=None, bold_max=None, index=False, bold_row_min=None):
    """Write <name>.csv and <name>.tex (booktabs). bold_min/bold_max: list of
    columns in which the best value is set in bold."""
    df = df.reset_index() if index else df.copy()
    df.to_csv(TAB_DIR / f"{name}.csv", index=False)
    best = {}
    for c in (bold_min or []):
        if c in df: best[c] = pd.to_numeric(df[c], errors="coerce").min()
    for c in (bold_max or []):
        if c in df: best[c] = pd.to_numeric(df[c], errors="coerce").max()
    cols = list(df.columns)
    align = "".join("l" if not np.issubdtype(df[c].dtype, np.number) else "r" for c in cols)
    lines = [r"\begin{table}[t]", r"\centering", r"\small"]
    if caption: lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label or 'tab:' + name}}}")
    lines += [rf"\begin{{tabular}}{{{align}}}", r"\toprule",
              " & ".join(_fmt(c, nd) for c in cols) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        cells = []
        rmin = pd.to_numeric(r[bold_row_min], errors="coerce").min() if bold_row_min else None
        for c in cols:
            s = _fmt(r[c], nd)
            if np.issubdtype(df[c].dtype, np.integer): s = str(int(r[c]))
            if bold_row_min and c in bold_row_min and np.isfinite(r[c]) and np.isclose(r[c], rmin):
                s = rf"\textbf{{{s}}}"
            elif c in best and isinstance(r[c], (float, np.floating, int, np.integer)) and np.isfinite(r[c]) \
               and np.isclose(r[c], best[c]):
                s = rf"\textbf{{{s}}}"
            cells.append(s)
        lines.append(" & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (TAB_DIR / f"{name}.tex").write_text("\n".join(lines))
    print(f"[table] {name} -> {TAB_DIR}")
    try:
        display(df)
    except NameError:
        print(df.to_string())
    return df

# ---------------------------------------------------------------------------
# Result caching & checkpoints (everything resumable after a Colab disconnect)
# ---------------------------------------------------------------------------
def cached(name, fn, force=False):
    f = RES_DIR / f"{name}.pkl"
    if f.exists() and not force:
        with open(f, "rb") as fh: return pickle.load(fh)
    out = fn()
    with open(f, "wb") as fh: pickle.dump(out, fh)
    return out

def set_seed(s):
    import random
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass

# ---------------------------------------------------------------------------
# Feature tensor & rolling windows
#   channels: 0 = 100*log(P_t'/P_origin)   (filled at gather time)
#             1 = price-changed indicator  2 = observed (not carried-forward)
#             3,4 = day-of-week sin/cos     5 = log1p(days since last change)/5
#   targets : 100*log(P_{o+h}/P_o), h = 1..H, clipped to [-100, 100]
# ---------------------------------------------------------------------------
H_MAX, H_EVAL = 30, [1, 3, 7, 14, 30]
N_CH = 6

def build_features(prep, idx):
    lp = prep["logP"][idx]
    n, T = lp.shape
    chg = np.zeros_like(lp); chg[:, 1:] = (np.diff(lp, axis=1) != 0)
    obs = np.broadcast_to((~prep["cf_day"]).astype(np.float64), (n, T))
    dow = prep["dates"].dayofweek.to_numpy()
    s, c = np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7)
    dsc = np.zeros_like(lp)
    for t in range(1, T):
        dsc[:, t] = np.where(chg[:, t] > 0, 0, dsc[:, t - 1] + 1)
    F = np.stack([lp, chg, obs, np.broadcast_to(s, (n, T)), np.broadcast_to(c, (n, T)),
                  np.log1p(dsc) / 5], axis=-1).astype(np.float32)
    return F                                                   # (n, T, C); channel 0 = raw log-price

def origins_for(fold, part, L, H=H_MAX):
    """Forecast origins o (inclusive) for 'train' | 'val' | 'test'. Targets at
    o+1..o+H are masked beyond the segment end (no peeking)."""
    if part == "train":
        return np.arange(L - 1, fold["train_end"]), fold["train_end"]
    if part == "val":
        return np.arange(fold["val_start"] - 1, fold["val_end"]), fold["val_end"]
    return np.arange(fold["test_start"] - 1, fold["test_end"]), fold["test_end"]

def gather_windows(F, idx, origins, L, H, seg_end, cf_day, rows=None):
    """Vectorised, ORIGIN-MAJOR gather for products `rows` (local indices into F,
    default all) at forecast origins `origins`.
    Returns X (k, n, L, C), Y (k, n, H), M (k, n, H) and meta (k, n, 2) holding
    (global product id, origin day). Channel 0 of X is re-based to the origin."""
    rows = np.arange(F.shape[0]) if rows is None else np.asarray(rows)
    oo, rr = np.meshgrid(origins, rows, indexing="ij")                    # (k, n)
    tt = oo[..., None] + np.arange(-L + 1, 1)                              # (k, n, L)
    X = F[rr[..., None], np.maximum(tt, 0)].copy()
    base = F[rr, oo, 0]
    X[..., 0] = 100.0 * (X[..., 0] - base[..., None])
    hh = oo[..., None] + np.arange(1, H + 1)                               # (k, n, H)
    valid = hh <= seg_end
    hc = np.minimum(hh, F.shape[1] - 1)
    M = valid & ~cf_day[hc]
    Y = 100.0 * (F[rr[..., None], hc, 0] - base[..., None])
    Y = np.clip(np.where(M, Y, 0.0), -100, 100).astype(np.float32)
    meta = np.stack([np.asarray(idx)[rr], oo], -1)
    return X.astype(np.float32), Y, M, meta

# ---------------------------------------------------------------------------
# Evaluation in PRICE space. Predictions are 100*log-ratio vs the origin price.
# ---------------------------------------------------------------------------
def price_scale(prep, idx, fold):
    """MASE scale: mean |P_t - P_{t-1}| on the fold's training window."""
    P = prep["P"][idx, : fold["train_end"] + 1]
    return np.maximum(np.abs(np.diff(P, axis=1)).mean(axis=1), 1e-6)

def evaluate(pred, Y, M, meta, prep, fold, horizons=H_EVAL):
    """pred, Y, M: (..., H) arrays (any leading shape); meta (..., 2).
    Returns (summary per horizon, {h: per-product mean sAPE})."""
    H = Y.shape[-1]
    pred, Y, M, meta = pred.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H), meta.reshape(-1, 2)
    P0 = prep["P"][meta[:, 0], meta[:, 1]]
    gids = np.unique(meta[:, 0])
    scale = pd.Series(price_scale(prep, gids, fold), index=gids)
    sc = scale.loc[meta[:, 0]].to_numpy()
    rows = []
    per_prod = {}
    for h in horizons:
        m = M[:, h - 1]
        if m.sum() == 0: continue
        y = P0[m] * np.exp(Y[m, h - 1] / 100); yh = P0[m] * np.exp(pred[m, h - 1] / 100)
        e = yh - y; ae = np.abs(e)
        sape = 200 * ae / (np.abs(y) + np.abs(yh))
        chg = np.abs(Y[m, h - 1]) > 1e-9
        prod = meta[m, 0]
        df = pd.DataFrame(dict(prod=prod, ae=ae, sape=sape, se=ae**2, ape=100 * ae / y, sc_ae=ae / sc[m], y=y))
        g = df.groupby("prod")
        mase_p = g.sc_ae.mean()
        rows.append(dict(h=h, n=int(m.sum()), MAE=ae.mean(), RMSE=np.sqrt((ae**2).mean()),
                         MAPE=df.ape.mean(), sMAPE=sape.mean(), WAPE=100 * ae.sum() / y.sum(),
                         MASE=mase_p.mean(), MASE_med=mase_p.median(),
                         CC_sMAPE=sape[chg].mean() if chg.any() else np.nan,
                         change_rate=chg.mean()))
        per_prod[h] = g.sape.mean()
    return pd.DataFrame(rows), per_prod

def change_auprc(score, Y, M, h):
    """AUPRC for 'will the price differ from today's price at horizon h?'."""
    from sklearn.metrics import average_precision_score
    H = Y.shape[-1]; score, Y, M = score.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H)
    m = M[:, h - 1]; lab = np.abs(Y[m, h - 1]) > 1e-9
    if lab.sum() == 0 or lab.all(): return np.nan
    return average_precision_score(lab, score[m, h - 1])

# ---------------------------------------------------------------------------
# Statistical tests
# ---------------------------------------------------------------------------
def dm_test(l1, l2, h=1):
    """Diebold-Mariano with Harvey-Leybourne-Newbold small-sample correction.
    l1, l2: loss series (e.g. cross-sectional mean sAPE per origin). Negative
    statistic => model 1 more accurate."""
    d = np.asarray(l1) - np.asarray(l2); d = d[np.isfinite(d)]
    T = len(d)
    if T < 5: return np.nan, np.nan
    dbar = d.mean(); g = [np.mean((d[k:] - dbar) * (d[: T - k] - dbar)) for k in range(min(h, T - 1))]
    v = (g[0] + 2 * sum(g[1:])) / T
    if v <= 0: return np.nan, np.nan
    stat = dbar / np.sqrt(v) * np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    return stat, 2 * sps.t.sf(abs(stat), df=T - 1)

def wilcoxon_paired(a, b):
    a, b = pd.Series(a), pd.Series(b); j = a.index.intersection(b.index)
    d = (a[j] - b[j]).to_numpy()
    if np.allclose(d, 0): return np.nan
    return sps.wilcoxon(d, zero_method="zsplit").pvalue

def bootstrap_ci(d, B=2000, seed=0, alpha=0.05):
    d = np.asarray(d); d = d[np.isfinite(d)]
    rng = np.random.default_rng(seed)
    bs = rng.choice(d, (B, len(d)), replace=True).mean(axis=1)
    return d.mean(), np.quantile(bs, alpha / 2), np.quantile(bs, 1 - alpha / 2)

def holm(pvals):
    p = np.asarray(pvals, dtype=float); out = np.full_like(p, np.nan)
    ok = np.isfinite(p); idx = np.argsort(p[ok]); m = ok.sum()
    adj = np.maximum.accumulate((m - np.arange(m)) * p[ok][idx]); adj = np.minimum(adj, 1)
    tmp = np.empty(m); tmp[idx] = adj; out[ok] = tmp
    return out

def per_origin_loss(pred, Y, M, meta, prep, h):
    """Cross-sectional mean sAPE per origin at horizon h (DM-test input)."""
    H = Y.shape[-1]
    pred, Y, M, meta = pred.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H), meta.reshape(-1, 2)
    m = M[:, h - 1]
    P0 = prep["P"][meta[m, 0], meta[m, 1]]
    y = P0 * np.exp(Y[m, h - 1] / 100); yh = P0 * np.exp(pred[m, h - 1] / 100)
    s = pd.Series(200 * np.abs(yh - y) / (y + yh)).groupby(meta[m, 1]).mean()
    return s

# ---------------------------------------------------------------------------
# Dataset description (figure + table used in the Data section of every paper)
# ---------------------------------------------------------------------------
def dataset_overview(prep):
    meta, dates = prep["meta"], prep["dates"]
    rows = [("Products (SKUs)", len(meta)), ("Days", len(dates)),
            ("Period", f"{dates[0].date()} to {dates[-1].date()}"),
            ("Food SKUs (tagged)", int(meta.is_food.sum())),
            ("Excluded: variant breaks", int(meta.variant_break.sum())),
            ("Scraper-outage days (carried forward)", int(prep["cf_day"].sum())),
            ("Platform batch-update days", int(prep["batch_day"].sum())),
            ("Spike-and-revert repairs", int(meta.spike_repairs.sum())),
            ("T0 (food, never changes in train)", int(((meta.tier == "T0") & meta.is_food).sum())),
            ("T1 (food, 1-9 changes)", int((meta.tier == "T1").sum())),
            ("T2 (food, >=10 changes)", int((meta.tier == "T2").sum()))]
    save_table(pd.DataFrame(rows, columns=["Item", "Value"]).astype({"Value": str}), "data_summary",
               "Dataset summary after quality control.")
    fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.2))
    ch = prep["daily_changes"]
    ax[0].bar(dates, ch, width=1.0, color=PALETTE[0], lw=0)
    ax[0].bar(dates[prep["batch_day"]], ch[prep["batch_day"]], width=2.5, color=PALETTE[1], lw=0, label="batch update")
    ax[0].scatter(dates[prep["cf_day"]], np.zeros(prep["cf_day"].sum()), marker="|", s=30, color=GREY, label="outage (0 changes)")
    ax[0].set_title("(a) Daily number of price changes"); ax[0].legend(loc="upper left")
    ax[0].tick_params(axis="x", rotation=30)
    nc = meta.loc[meta.is_food, "n_changes_train"]
    ax[1].hist(np.log10(nc + 1), bins=30, color=PALETTE[0], edgecolor="white", lw=0.4)
    ax[1].set_xlabel("log$_{10}$(1 + price changes in training window)"); ax[1].set_ylabel("Food SKUs")
    ax[1].set_title("(b) Price stickiness")
    act = meta[meta.tier.isin(["T1", "T2"])]
    order = act.category.value_counts()
    ax[2].barh(order.index[::-1], order.values[::-1], color=PALETTE[0])
    ax[2].set_title("(c) Active food SKUs by category"); ax[2].tick_params(axis="y", labelsize=6)
    fig.tight_layout(); savefig(fig, "fig_data_overview")

# ---------------------------------------------------------------------------
# Fold data bundle: windows for train / val / test in one call (cached in RAM)
# ---------------------------------------------------------------------------
def fold_data(prep, F, idx, fold, L, H=H_MAX, parts=("train", "val", "test"), train_stride=1):
    out = {}
    for part in parts:
        O, seg_end = origins_for(fold, part, L, H)
        if part == "train": O = O[::train_stride]
        out[part] = gather_windows(F, idx, O, L, H, seg_end, prep["cf_day"])
    return out

def naive_family(X, H=H_MAX):
    """Naive (0), seasonal-naive(7) and moving-average(7) forecasts from windows.
    X channel 0 is 100*log(P/P_origin) so the naive forecast is identically 0."""
    k = X[..., 0]
    naive = np.zeros(k.shape[:-1] + (H,), np.float32)
    L = k.shape[-1]
    snaive = np.stack([k[..., L - 7 + ((h - 1) % 7)] for h in range(1, H + 1)], -1)
    ma = np.repeat(k[..., -7:].mean(-1, keepdims=True), H, -1)
    return {"Naive": naive, "SNaive-7": snaive.astype(np.float32), "MA-7": ma.astype(np.float32)}

def results_frame(res_dict):
    """{model: summary-df} -> long frame with a 'model' column."""
    return pd.concat([d.assign(model=m) for m, d in res_dict.items()], ignore_index=True)

In [ ]:
# ============================================================================
# SECTION 1c — SHARED PYTORCH UTILITIES: resumable trainer + building blocks
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as Fnn
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print("device:", DEVICE)

def masked_huber(pred, y, m, delta=1.0):
    m = m.float()
    l = Fnn.huber_loss(pred, y, reduction="none", delta=delta)
    return (l * m).sum() / m.sum().clamp(min=1.0)

def iterate_flat(X, Y, M, bs, shuffle, rng=None, extra=None):
    """Mini-batches over flattened (origin, product) windows. X: (k,n,L,C).
    extra: dict of (k,n,...) arrays gathered alongside (e.g. category ids)."""
    k, n = X.shape[:2]
    N = k * n
    order = (rng or np.random.default_rng(0)).permutation(N) if shuffle else np.arange(N)
    Xf, Yf, Mf = X.reshape(N, *X.shape[2:]), Y.reshape(N, -1), M.reshape(N, -1)
    ex = {kk: v.reshape(N, *v.shape[2:]) for kk, v in (extra or {}).items()}
    for s in range(0, N, bs):
        j = order[s: s + bs]
        b = dict(x=torch.from_numpy(Xf[j]).to(DEVICE), y=torch.from_numpy(Yf[j]).to(DEVICE),
                 m=torch.from_numpy(Mf[j]).to(DEVICE))
        for kk, v in ex.items(): b[kk] = torch.from_numpy(v[j]).to(DEVICE)
        yield b

def iterate_snap(X, Y, M, bs, shuffle, rng=None, extra=None):
    """Mini-batches of whole cross-sections (all products at an origin) for
    graph / cross-product models. X: (k, n, L, C) -> batch x: (b, n, L, C)."""
    k = X.shape[0]
    order = (rng or np.random.default_rng(0)).permutation(k) if shuffle else np.arange(k)
    for s in range(0, k, bs):
        j = order[s: s + bs]
        b = dict(x=torch.from_numpy(X[j]).to(DEVICE), y=torch.from_numpy(Y[j]).to(DEVICE),
                 m=torch.from_numpy(M[j]).to(DEVICE))
        for kk, v in (extra or {}).items(): b[kk] = torch.from_numpy(v[j]).to(DEVICE)
        yield b

def fit(model, train_iter, val_iter, loss_fn, ckpt_path, epochs=30, lr=1e-3, wd=1e-4,
        patience=6, verbose=True, clip=1.0):
    """Resumable training loop. `train_iter(epoch)` / `val_iter()` yield batches,
    `loss_fn(model, batch)` returns a scalar loss. A checkpoint is written after
    every epoch (model, optimiser, scheduler, history, early-stopping state);
    re-running the cell resumes where it stopped; a finished run is skipped."""
    ckpt_path = Path(ckpt_path); ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    st = dict(epoch=0, best=np.inf, bad=0, hist=[], done=False, best_state=None)
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        st = ck["state"]
        if st["done"]:
            model.load_state_dict(st["best_state"]); model.eval()
            return model, pd.DataFrame(st["hist"])
        if verbose: print(f"  resuming {ckpt_path.name} at epoch {st['epoch']}")
    for ep in range(st["epoch"], epochs):
        model.train(); t0 = time.time(); tl, nb = 0.0, 0
        for b in train_iter(ep):
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
                loss = loss_fn(model, b)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update(); tl += loss.item(); nb += 1
        sched.step()
        model.eval(); vl, vb = 0.0, 0
        with torch.no_grad():
            for b in val_iter():
                with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
                    vl += loss_fn(model, b).item(); vb += 1
        vl /= max(vb, 1); tl /= max(nb, 1)
        st["hist"].append(dict(epoch=ep, train=tl, val=vl, sec=time.time() - t0))
        if vl < st["best"] - 1e-5:
            st["best"], st["bad"] = vl, 0
            st["best_state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            st["bad"] += 1
        st["epoch"] = ep + 1
        stop = st["bad"] >= patience or st["epoch"] >= epochs
        st["done"] = stop
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), state=st), ckpt_path)
        if verbose: print(f"  ep {ep:02d} train {tl:.4f} val {vl:.4f} ({time.time()-t0:.1f}s)")
        if stop: break
    if st["best_state"] is not None: model.load_state_dict(st["best_state"])
    model.eval()
    return model, pd.DataFrame(st["hist"])

@torch.no_grad()
def predict(model, batches, key="x", fwd=None):
    model.eval(); out = []
    for b in batches:
        with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
            p = fwd(model, b) if fwd else model(b[key])
        out.append(p.float().cpu().numpy())
    return np.concatenate(out, 0)

# ---------------------------------------------------------------------------
# Building blocks shared by several papers
# ---------------------------------------------------------------------------
class PatchEncoder(nn.Module):
    """PatchTST-style temporal encoder: (B, L, C) -> (B, d)."""
    def __init__(self, L, C, d=64, patch=6, layers=2, heads=4, dropout=0.1):
        super().__init__()
        assert L % patch == 0
        self.patch, self.np = patch, L // patch
        self.proj = nn.Linear(patch * C, d)
        self.pos = nn.Parameter(torch.randn(1, self.np, d) * 0.02)
        el = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(el, layers, enable_nested_tensor=False)
        self.out = nn.Sequential(nn.LayerNorm(2 * d), nn.Linear(2 * d, d))
    def tokens(self, x):
        B, L, C = x.shape
        z = x.reshape(B, self.np, self.patch * C)
        return self.enc(self.proj(z) + self.pos)                 # (B, np, d)
    def forward(self, x):
        h = self.tokens(x)
        return self.out(torch.cat([h.mean(1), h[:, -1]], -1))

class GRUEncoder(nn.Module):
    def __init__(self, C, d=64, layers=1):
        super().__init__(); self.rnn = nn.GRU(C, d, layers, batch_first=True)
    def forward(self, x): return self.rnn(x)[0][:, -1]

class LSTMEncoder(nn.Module):
    def __init__(self, C, d=64, layers=2, dropout=0.1):
        super().__init__(); self.rnn = nn.LSTM(C, d, layers, batch_first=True, dropout=dropout)
    def forward(self, x): return self.rnn(x)[0][:, -1]

class MLPEncoder(nn.Module):
    def __init__(self, L, C, d=64):
        super().__init__(); self.net = nn.Sequential(nn.Flatten(), nn.Linear(L * C, 2 * d), nn.GELU(), nn.Linear(2 * d, d))
    def forward(self, x): return self.net(x)

class Head(nn.Module):
    def __init__(self, d, H, dropout=0.1):
        super().__init__(); self.net = nn.Sequential(nn.LayerNorm(d), nn.Dropout(dropout), nn.Linear(d, d), nn.GELU(), nn.Linear(d, H))
    def forward(self, h): return self.net(h)

In [ ]:
# ==== 1d. BUILD / LOAD THE CLEAN PANEL (cached on Drive) ====================
prep = build_prep(DATA_PATH, PREP_DIR)
meta, FOLDS = prep["meta"], prep["folds"]
CATS = sorted(meta.category.unique()); CAT2ID = {c: i for i, c in enumerate(CATS)}
for f in FOLDS:
    d = prep["dates"]
    print(f"fold {f['fold']}: train ..{d[f['train_end']].date()} | val {d[f['val_start']].date()}..{d[f['val_end']].date()}"
          f" | test {d[f['test_start']].date()}..{d[f['test_end']].date()}")
print(meta.tier.value_counts().to_dict())

In [ ]:
# ==== 2. DATASET OVERVIEW (figure + table for the Data section) =============
dataset_overview(prep)

## 3. Experimental configuration
Node universe: all *active* food SKUs (tiers T1 and T2). Input window L = 60 days, horizons 1..30
(reported at 1, 3, 7, 14, 30). In `SMOKE` mode everything is shrunk so the notebook finishes in minutes.

In [ ]:
# ==== 3. CONFIGURATION ======================================================
L, H = 60, H_MAX
IDX = subset_idx(prep, ("T1", "T2"), smoke_n=60 if SMOKE else None)
N = len(IDX)
CAT = np.array([CAT2ID[c] for c in meta.category.to_numpy()[IDX]])
F = build_features(prep, IDX)
FOLD_IDS = [1, 2] if SMOKE else [0, 1, 2]
SEEDS = [0] if SMOKE else [0, 1, 2]            # seeds for main models & ablations
FACT_SEEDS = [0] if SMOKE else [0, 1]          # seeds for the factorial experiment
EPOCHS = 2 if SMOKE else 40
PATIENCE = 6
K_GRAPH = 10                                   # neighbours per node in every constructed graph
TRAIN_STRIDE = 1
print(f"N={N} products | folds {FOLD_IDS} | seeds {SEEDS} | epochs {EPOCHS} | device {DEVICE}")

D = {f: fold_data(prep, F, IDX, FOLDS[f], L, train_stride=TRAIN_STRIDE) for f in FOLD_IDS}
for f in FOLD_IDS:
    print(f"fold {f}: " + ", ".join(f"{p} {D[f][p][0].shape[:2]}" for p in ("train", "val", "test")))
PRED_DIR = RES_DIR / "preds"; PRED_DIR.mkdir(exist_ok=True)

def cat_extra(X):
    return {"cat": np.ascontiguousarray(np.broadcast_to(CAT[None], X.shape[:2]))}

## 4. Baselines
### 4.1 Naive family and statistical models
Naive (random walk), seasonal naive (7 days), 7-day moving average, AutoETS and AutoARIMA
(`statsforecast`). Statistical models are fitted once on each fold's training window and then *filtered
forward* (no refit) through validation and test data, producing a 30-day forecast at every test origin.

In [ ]:
# ==== 4.1 NAIVE FAMILY + AutoETS / AutoARIMA ================================
from joblib import Parallel, delayed

def _stat_one(y, train_end, origins, H, kind):
    from statsforecast.models import AutoARIMA, AutoETS
    try:
        mdl = (AutoARIMA(season_length=7) if kind == "AutoARIMA" else AutoETS(season_length=7)).fit(y[: train_end + 1])
        out = np.stack([mdl.forward(y=y[: o + 1], h=H)["mean"] - y[o] for o in origins])
    except Exception:
        out = np.zeros((len(origins), H))
    return 100 * np.clip(out, -1, 1)

def stat_forecast(kind, f):
    fold = FOLDS[f]; O, _ = origins_for(fold, "test", L)
    lp = prep["logP"][IDX]
    def run():
        res = Parallel(n_jobs=-1)(delayed(_stat_one)(lp[i], fold["train_end"], O, H, kind) for i in range(N))
        return np.stack(res, 1).astype(np.float32)             # (k, n, H)
    return cached(f"stat_{kind}_fold{f}_N{N}", run)

PRED = {f: {} for f in FOLD_IDS}                               # PRED[fold][model] -> list of seed predictions
for f in FOLD_IDS:
    for m, p in naive_family(D[f]["test"][0]).items(): PRED[f][m] = [p]
    for kind in ("AutoETS", "AutoARIMA"):
        t0 = time.time(); PRED[f][kind] = [stat_forecast(kind, f)]
        print(f"fold {f} {kind}: {time.time()-t0:.1f}s")

### 4.2 Gradient boosting (LightGBM, direct multi-horizon)
One global LightGBM per reported horizon on hand-crafted window features (lags, rolling moments over
7/14/30/60 days, recent change counts, days since the last change, calendar, category). Early stopping uses
the validation window.

In [ ]:
# ==== 4.2 LightGBM ==========================================================
import lightgbm as lgb

def tab_features(X, cat):
    v = X[..., 0]; c = X[..., 1]
    feats = [v[..., -30:-1][..., ::-1]]
    for w in (7, 14, 30, 60):
        feats += [v[..., -w:].mean(-1, keepdims=True), v[..., -w:].std(-1, keepdims=True)]
    feats += [c[..., -7:].sum(-1, keepdims=True), c[..., -30:].sum(-1, keepdims=True), c.sum(-1, keepdims=True),
              X[..., -1, 5:6], X[..., -1, 3:5], X[..., -1, 2:3], cat[..., None].astype(np.float32)]
    Z = np.concatenate(feats, -1)
    return Z.reshape(-1, Z.shape[-1])

def lgbm_forecast(f, seed=0, max_rows=400_000):
    def run():
        Xtr, Ytr, Mtr, _ = D[f]["train"]; Xva, Yva, Mva, _ = D[f]["val"]; Xte = D[f]["test"][0]
        Ztr = tab_features(Xtr, cat_extra(Xtr)["cat"]); Zva = tab_features(Xva, cat_extra(Xva)["cat"])
        Zte = tab_features(Xte, cat_extra(Xte)["cat"])
        out = np.zeros(Xte.shape[:2] + (H,), np.float32)
        rng = np.random.default_rng(seed)
        for h in H_EVAL:
            mtr = Mtr[..., h - 1].ravel(); mva = Mva[..., h - 1].ravel()
            j = np.where(mtr)[0]
            if len(j) > max_rows: j = rng.choice(j, max_rows, replace=False)
            reg = lgb.LGBMRegressor(n_estimators=100 if SMOKE else 2000, learning_rate=0.03, num_leaves=63,
                                    min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                    objective="huber", alpha=1.0, random_state=seed, verbose=-1)
            reg.fit(Ztr[j], Ytr[..., h - 1].ravel()[j], eval_set=[(Zva[mva], Yva[..., h - 1].ravel()[mva])],
                    callbacks=[lgb.early_stopping(100, verbose=False)])
            out[..., h - 1] = reg.predict(Zte).reshape(Xte.shape[:2])
        return out
    return cached(f"lgbm_fold{f}_s{seed}_N{N}", run)

for f in FOLD_IDS:
    PRED[f]["LightGBM"] = [lgbm_forecast(f)]
print("LightGBM done")

### 4.3 Deep univariate baselines (global, channel-independent)
LSTM, PatchTST and a compact Temporal Fusion Transformer (TFT-lite: gated residual networks, variable
selection over the six input channels, static category context, LSTM encoder and interpretable multi-head
attention). All three are trained as *global* models over every product's windows with a masked Huber loss.

In [ ]:
# ==== 4.3 DEEP UNIVARIATE MODELS ============================================
class UniForecaster(nn.Module):
    def __init__(self, enc, n_cat, d=64, H=H_MAX):
        super().__init__(); self.enc, self.cat, self.head = enc, nn.Embedding(n_cat, d), Head(d, H)
    def forward(self, x, cat): return self.head(self.enc(x) + self.cat(cat))

class GRN(nn.Module):
    def __init__(self, d_in, d, d_out=None, d_ctx=None, dropout=0.1):
        super().__init__(); d_out = d_out or d
        self.fc1 = nn.Linear(d_in, d); self.ctx = nn.Linear(d_ctx, d, bias=False) if d_ctx else None
        self.fc2 = nn.Linear(d, 2 * d_out); self.skip = nn.Linear(d_in, d_out) if d_in != d_out else nn.Identity()
        self.drop = nn.Dropout(dropout); self.ln = nn.LayerNorm(d_out)
    def forward(self, x, c=None):
        h = self.fc1(x)
        if self.ctx is not None and c is not None: h = h + self.ctx(c)
        h = self.drop(Fnn.glu(self.fc2(Fnn.elu(h)), -1))
        return self.ln(self.skip(x) + h)

class TFTLite(nn.Module):
    def __init__(self, C, n_cat, d=64, H=H_MAX, heads=4):
        super().__init__()
        self.C = C; self.var = nn.ModuleList([nn.Linear(1, d) for _ in range(C)])
        self.var_grn = nn.ModuleList([GRN(d, d) for _ in range(C)])
        self.static = nn.Embedding(n_cat, d); self.cs = GRN(d, d); self.ch = GRN(d, d); self.cc = GRN(d, d)
        self.vsn = GRN(C * d, d, C, d_ctx=d)
        self.lstm = nn.LSTM(d, d, batch_first=True); self.gate = GRN(d, d); self.enrich = GRN(d, d, d_ctx=d)
        self.attn = nn.MultiheadAttention(d, heads, dropout=0.1, batch_first=True)
        self.post = GRN(d, d); self.out = nn.Linear(d, H); self.last_vsn = None
    def forward(self, x, cat):
        s = self.static(cat); cs = self.cs(s)
        e = [self.var[i](x[..., i: i + 1]) for i in range(self.C)]              # C x (B, L, d)
        w = torch.softmax(self.vsn(torch.cat(e, -1), cs[:, None]), -1)          # (B, L, C)
        v = torch.stack([self.var_grn[i](e[i]) for i in range(self.C)], -1)     # (B, L, d, C)
        z = (v * w[:, :, None, :]).sum(-1)
        self.last_vsn = w.mean(1).detach()
        h, _ = self.lstm(z, (self.ch(s)[None].to(z.dtype).contiguous(), self.cc(s)[None].to(z.dtype).contiguous()))
        h = self.gate(h + z); h = self.enrich(h, cs[:, None])
        a, _ = self.attn(h[:, -1:], h, h)
        return self.out(self.post(a[:, 0] + h[:, -1]))

def loss_flat(model, b): return masked_huber(model(b["x"], b["cat"]), b["y"], b["m"])

def run_flat(name, build, f, seed, bs=512, lr=1e-3):
    key = f"{name}__f{f}__s{seed}"; pf = PRED_DIR / f"{key}.npz"
    if pf.exists(): return np.load(pf)["pred"].astype(np.float32)
    set_seed(seed); model = build()
    tr, va, te = D[f]["train"], D[f]["val"], D[f]["test"]
    rng = np.random.default_rng(seed)
    model, hist = fit(model, lambda ep: iterate_flat(*tr[:3], bs, True, rng, cat_extra(tr[0])),
                      lambda: iterate_flat(*va[:3], 4 * bs, False, extra=cat_extra(va[0])),
                      loss_flat, CKPT_DIR / f"{key}.pt", EPOCHS, lr, patience=PATIENCE, verbose=False)
    pred = predict(model, iterate_flat(*te[:3], 4 * bs, False, extra=cat_extra(te[0])),
                   fwd=lambda m, b: m(b["x"], b["cat"])).reshape(te[1].shape)
    np.savez_compressed(pf, pred=pred.astype(np.float16), val=hist.val.min())
    return pred

UNI = {"LSTM": lambda: UniForecaster(LSTMEncoder(N_CH), len(CATS)),
       "PatchTST": lambda: UniForecaster(PatchEncoder(L, N_CH), len(CATS)),
       "TFT-lite": lambda: TFTLite(N_CH, len(CATS))}
for name, build in UNI.items():
    for f in FOLD_IDS:
        t0 = time.time()
        PRED[f][name] = [run_flat(name, build, f, s) for s in SEEDS]
        print(f"{name} fold {f}: {time.time()-t0:.1f}s")

## 5. Product-graph construction (G0-G8)
Every graph is estimated on the fold's **training window only**, after removing scraper-outage, gap-end and
batch-update days, and is sparsified to the top-k = 10 neighbours per node, so graphs differ only in *which*
edges they contain, not in how many.

| Graph | Construction |
|---|---|
| G0 | no edges (self only) — no cross-product information |
| G1 | random, degree-matched (control) |
| G2 | random neighbours within the same tagged category |
| G3 | product-name similarity (TF-IDF over character 3-5-grams, cosine) |
| G4 | contemporaneous Pearson correlation of daily log-returns |
| G4-raw | as G4 but *including* batch-update days (contamination check) |
| G5 | lagged cross-correlation, max over lags 1-7 (directed, leader to follower) |
| G6 | mutual information between price-move directions (down / none / up) |
| G7 | co-change Jaccard similarity of price-change days (+-1 day tolerance) |
| G8 | fully learned adjacency (node embeddings, no prior) |

In [ ]:
# ==== 5. GRAPH CONSTRUCTION =================================================
from sklearn.feature_extraction.text import TfidfVectorizer

def _topk(S, k, rng):
    S = np.nan_to_num(S.astype(np.float64), nan=0.0) + rng.uniform(0, 1e-9, S.shape)   # random tie-break
    np.fill_diagonal(S, -np.inf)
    k = min(k, S.shape[0] - 1)
    nb = np.argpartition(-S, k, axis=1)[:, :k]
    A = np.zeros_like(S); r = np.arange(S.shape[0])[:, None]
    A[r, nb] = np.maximum(S[r, nb], 0) + 1e-3
    return A

def _standardise(R, valid):
    Z = np.where(valid[None], R, np.nan)
    mu, sd = np.nanmean(Z, 1, keepdims=True), np.nanstd(Z, 1, keepdims=True)
    Z = (Z - mu) / np.where(sd > 0, sd, np.inf)
    return np.nan_to_num(Z), valid.sum()

def build_graphs(f, k=K_GRAPH, seed=0):
    def run():
        rng = np.random.default_rng(seed); fold = FOLDS[f]; te = fold["train_end"]
        R = np.diff(prep["logP"][IDX, : te + 1], axis=1)                     # return on day t = column t-1
        cf, ge, bd = prep["cf_day"][1: te + 1], prep["gap_end"][1: te + 1], prep["batch_day"][1: te + 1]
        valid = ~(cf | ge | bd); valid_raw = ~(cf | ge)
        G = {"G0": np.zeros((N, N))}
        G["G1"] = _topk(rng.random((N, N)), k, rng)
        same = CAT[:, None] == CAT[None, :]
        G["G2"] = _topk(np.where(same, rng.random((N, N)), -1.0), k, rng) * same
        tf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1).fit_transform(
            [prep["names"][i].lower() for i in IDX])
        G["G3"] = _topk((tf @ tf.T).toarray(), k, rng)
        Z, n = _standardise(R, valid); C4 = Z @ Z.T / max(n, 1)
        G["G4"] = _topk(C4, k, rng)
        Zr, nr = _standardise(R, valid_raw); G["G4-raw"] = _topk(Zr @ Zr.T / max(nr, 1), k, rng)
        lag = np.full((N, N), -np.inf)
        for l in range(1, 8):
            Cl = Z[:, l:] @ Z[:, :-l].T / max(n - l, 1)                      # corr(r_i[t], r_j[t-l])
            lag = np.maximum(lag, Cl)
        G["G5"] = _topk(lag, k, rng)
        S = np.sign(np.where(valid[None], R, 0.0)).astype(np.int8)
        oh = np.stack([(S == v) for v in (-1, 0, 1)], 0).astype(np.float32)[:, :, valid]   # (3, N, T')
        T_ = oh.shape[-1]; pa = oh.mean(-1)                                  # (3, N)
        MI = np.zeros((N, N))
        for a in range(3):
            for b in range(3):
                pab = oh[a] @ oh[b].T / T_
                with np.errstate(divide="ignore", invalid="ignore"):
                    MI += np.nan_to_num(pab * np.log(pab / (pa[a][:, None] * pa[b][None, :])))
        G["G6"] = _topk(MI, k, rng)
        Cc = (np.abs(R) > 0) & valid[None]
        Cd = Cc.copy(); Cd[:, 1:] |= Cc[:, :-1]; Cd[:, :-1] |= Cc[:, 1:]
        Cc, Cd = Cc.astype(np.float32), Cd.astype(np.float32)
        inter = Cc @ Cd.T; union = Cc.sum(1)[:, None] + Cc.sum(1)[None, :] - np.minimum(inter, Cc @ Cc.T)
        G["G7"] = _topk(inter / np.maximum(union, 1), k, rng)
        G["_corr"] = C4                                                      # kept for analyses
        return {g: A.astype(np.float32) for g, A in G.items()}
    return cached(f"graphs_fold{f}_N{N}_k{k}", run)

GRAPHS = {f: build_graphs(f) for f in FOLD_IDS}
GRAPH_NAMES = ["G0", "G1", "G2", "G3", "G4", "G4-raw", "G5", "G6", "G7", "G8"]
print("graphs:", [g for g in GRAPHS[FOLD_IDS[-1]] if not g.startswith("_")])

In [ ]:
# ==== 5.1 GRAPH STATISTICS (table + figures) ================================
def edge_set(A): return set(zip(*np.nonzero(A)))
f_ref = FOLD_IDS[-1]; Gr = GRAPHS[f_ref]; C4 = Gr["_corr"]
rows = []
for g in GRAPH_NAMES[1:-1]:
    A = Gr[g] > 0; e = np.nonzero(A)
    rows.append(dict(graph=g, edges=int(A.sum()), mean_degree=A.sum(1).mean(),
                     homophily=(CAT[e[0]] == CAT[e[1]]).mean(),
                     mean_edge_corr=C4[e].mean(), reciprocity=(A & A.T).sum() / max(A.sum(), 1),
                     jaccard_vs_G4=len(edge_set(A) & edge_set(Gr["G4"])) / max(len(edge_set(A) | edge_set(Gr["G4"])), 1)))
save_table(pd.DataFrame(rows), "tab_graph_statistics",
           "Properties of the constructed product graphs (latest fold, training window). Homophily = share of "
           "edges linking products of the same category; mean edge correlation = mean training-window return "
           "correlation across an edge.", nd=3)

gl = GRAPH_NAMES[1:-1]
J = np.array([[len(edge_set(Gr[a] > 0) & edge_set(Gr[b] > 0)) / max(len(edge_set(Gr[a] > 0) | edge_set(Gr[b] > 0)), 1)
               for b in gl] for a in gl])
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.6))
im = ax[0].imshow(J, cmap="Blues", vmin=0, vmax=1)
ax[0].set_xticks(range(len(gl)), gl, rotation=45); ax[0].set_yticks(range(len(gl)), gl); ax[0].grid(False)
for i in range(len(gl)):
    for j in range(len(gl)):
        ax[0].text(j, i, f"{J[i,j]:.2f}", ha="center", va="center", fontsize=5.5, color="white" if J[i, j] > 0.6 else "#0b0b0b")
ax[0].set_title("(a) Edge overlap (Jaccard) between graphs"); fig.colorbar(im, ax=ax[0], shrink=0.8)
hom = pd.DataFrame(rows).set_index("graph")["homophily"]
ax[1].bar(hom.index, hom.values, color=PALETTE[0]); ax[1].set_ylabel("Same-category edge share")
ax[1].set_title("(b) Category homophily"); ax[1].tick_params(axis="x", rotation=45)
fig.tight_layout(); savefig(fig, "fig_graph_overlap_homophily")

# batch-day contamination: edges that exist only because of platform batch updates
only_raw = edge_set(Gr["G4-raw"] > 0) - edge_set(Gr["G4"] > 0)
contam = pd.DataFrame([dict(quantity="G4-raw edges", value=int((Gr['G4-raw'] > 0).sum())),
                       dict(quantity="edges lost when batch days are removed", value=len(only_raw)),
                       dict(quantity="share of G4-raw edges driven by batch days", value=len(only_raw) / max((Gr['G4-raw'] > 0).sum(), 1)),
                       dict(quantity="batch-update days in training window", value=int(prep['batch_day'][: FOLDS[f_ref]['train_end'] + 1].sum()))])
save_table(contam, "tab_batch_contamination", "Effect of platform batch-update days on the correlation graph.", nd=3)

## 6. Graph models
* **GCN-GRU** — GRU temporal encoder, two graph-convolution layers with symmetric normalised adjacency.
* **GAT-Transformer** — PatchTST encoder, two masked multi-head graph-attention layers over the graph's edges.
* **DGTT (proposed)** — PatchTST encoder, product and category embeddings, and two *dynamic graph layers*.
  For origin t, the attention logit from product i to product j is

  s_ij(t) = q_i(t)^T k_j(t) / sqrt(d)  [dynamic]  +  e_i^T e'_j  [learned static]  +  beta log(A_ij + eps)  [statistical prior]

  Only the top-k logits per row are kept (k = 16), the rest are masked, and messages are aggregated with a
  softmax. The adjacency therefore changes with every forecast origin.
* **iTransformer** — each product is one token; full attention across all products (dense cross-product
  baseline without any graph).

In [ ]:
# ==== 6. GRAPH / CROSS-PRODUCT MODELS =======================================
class DynamicGraphLayer(nn.Module):
    def __init__(self, N, d, heads=4, k=16, dynamic=True, static=True, prior=None, graph_attn=True, dropout=0.1):
        super().__init__()
        self.h, self.dh, self.k = heads, d // heads, k
        self.dynamic, self.static, self.graph_attn = dynamic, static, graph_attn
        self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * d, d))
        self.drop = nn.Dropout(dropout)
        if static:
            self.e1 = nn.Parameter(torch.randn(N, 16) * 0.1); self.e2 = nn.Parameter(torch.randn(N, 16) * 0.1)
        if prior is not None:
            P = torch.as_tensor(prior, dtype=torch.float32); P = P / P.sum(1, keepdim=True).clamp(min=1e-9)
            self.register_buffer("logprior", torch.log(P + 1e-4)); self.beta = nn.Parameter(torch.tensor(1.0))
        else:
            self.logprior = None
        self.last_attn = None
    def forward(self, x):                                            # x: (B, N, d)
        B, N, d = x.shape
        q, k, v = self.qkv(self.ln1(x)).view(B, N, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        logits = torch.zeros(B, self.h, N, N, device=x.device, dtype=q.dtype)
        if self.dynamic: logits = logits + q @ k.transpose(-1, -2) / math.sqrt(self.dh)
        if self.static: logits = logits + (self.e1 @ self.e2.T).to(q.dtype)[None, None]
        if self.logprior is not None: logits = logits + (self.beta * self.logprior).to(q.dtype)[None, None]
        eye = torch.eye(N, device=x.device, dtype=torch.bool)
        logits = logits.masked_fill(eye, float("-inf"))
        kk = min(self.k, N - 1)
        thr = logits.topk(kk, dim=-1).values[..., -1:]
        keep = logits >= thr
        if self.graph_attn:
            w = torch.softmax(logits.masked_fill(~keep, float("-inf")), -1)
        else:
            w = keep.to(q.dtype) / keep.sum(-1, keepdim=True).clamp(min=1)
        self.last_attn = w.mean(1).detach()                          # (B, N, N)
        out = (w @ v).transpose(1, 2).reshape(B, N, d)
        x = x + self.drop(self.o(out))
        return x + self.drop(self.ff(self.ln2(x)))

class GraphForecaster(nn.Module):
    """arch in {'gcn','gat','dgtt'}; graph = (N,N) prior or None; learned=True -> learned adjacency (G8)."""
    def __init__(self, arch, graph=None, learned=False, d=64, H=H_MAX, k=16, temporal=None, prod_emb=True,
                 dynamic=True, static=True, use_prior=True, graph_attn=True, message_passing=True):
        super().__init__()
        self.arch, self.mp = arch, message_passing
        temporal = temporal or ("gru" if arch == "gcn" else "patch")
        self.enc = {"gru": lambda: GRUEncoder(N_CH, d), "patch": lambda: PatchEncoder(L, N_CH, d),
                    "mlp": lambda: MLPEncoder(L, N_CH, d)}[temporal]()
        self.pid = nn.Embedding(N, d) if prod_emb else None
        self.cat = nn.Embedding(len(CATS), d)
        self.register_buffer("catv", torch.as_tensor(CAT, dtype=torch.long))
        self.learned = learned
        if learned and arch != "dgtt":
            self.e1 = nn.Parameter(torch.randn(N, 16) * 0.1); self.e2 = nn.Parameter(torch.randn(N, 16) * 0.1)
        if arch == "gcn":
            if graph is not None:
                A = np.maximum(graph, graph.T) > 0
                A = A.astype(np.float32) + np.eye(N, dtype=np.float32)
                dinv = 1 / np.sqrt(A.sum(1)); self.register_buffer("Ahat", torch.as_tensor(dinv[:, None] * A * dinv[None]))
            self.lins = nn.ModuleList([nn.Linear(d, d) for _ in range(2)]); self.lns = nn.ModuleList([nn.LayerNorm(d) for _ in range(2)])
        elif arch == "gat":
            if graph is not None:
                self.register_buffer("mask", torch.as_tensor((graph > 0) | np.eye(N, dtype=bool)))
            self.attn = nn.ModuleList([nn.MultiheadAttention(d, 4, dropout=0.1, batch_first=True) for _ in range(2)])
            self.lns = nn.ModuleList([nn.LayerNorm(d) for _ in range(2)])
        else:
            prior = graph if (use_prior and graph is not None and graph.sum() > 0) else None
            self.layers = nn.ModuleList([DynamicGraphLayer(N, d, 4, k, dynamic, static, prior, graph_attn) for _ in range(2)])
        self.head = Head(d, H)
    def learned_adj(self, k=10):
        S = torch.relu(self.e1 @ self.e2.T); S = S.masked_fill(torch.eye(N, device=S.device, dtype=torch.bool), -1)
        thr = S.topk(min(k, N - 1), -1).values[:, -1:]
        return torch.where(S >= thr, torch.softmax(S.masked_fill(S < thr, float("-inf")), -1), torch.zeros_like(S))
    def forward(self, x):                                            # x: (B, N, L, C)
        B = x.shape[0]
        h = self.enc(x.reshape(B * N, L, N_CH)).view(B, N, -1) + self.cat(self.catv)[None]
        if self.pid is not None: h = h + self.pid.weight[None]
        if self.mp:
            if self.arch == "gcn":
                A = self.learned_adj() if self.learned else getattr(self, "Ahat", None)
                if A is not None:
                    for lin, ln in zip(self.lins, self.lns): h = ln(h + torch.relu(lin(A.to(h.dtype) @ h)))
            elif self.arch == "gat":
                if self.learned: mask = (self.learned_adj() > 0) | torch.eye(N, device=x.device, dtype=torch.bool)
                else: mask = getattr(self, "mask", torch.eye(N, device=x.device, dtype=torch.bool))
                for att, ln in zip(self.attn, self.lns):
                    a, _ = att(h, h, h, attn_mask=~mask); h = ln(h + a)
            else:
                for lay in self.layers: h = lay(h)
        return self.head(h)

class ITransformer(nn.Module):
    def __init__(self, d=64, H=H_MAX, layers=2):
        super().__init__()
        self.emb = nn.Sequential(nn.Linear(2 * L, d), nn.GELU(), nn.Linear(d, d)); self.cat = nn.Embedding(len(CATS), d)
        self.register_buffer("catv", torch.as_tensor(CAT, dtype=torch.long))
        el = nn.TransformerEncoderLayer(d, 4, 2 * d, 0.1, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(el, layers, enable_nested_tensor=False); self.head = Head(d, H)
    def forward(self, x):
        tok = self.emb(torch.cat([x[..., 0], x[..., 1]], -1)) + self.cat(self.catv)[None]
        return self.head(self.enc(tok))

def loss_snap(model, b): return masked_huber(model(b["x"]), b["y"], b["m"])

REG_FILE = RES_DIR / "run_registry.csv"
def run_snap(name, build, f, seed, bs=4, lr=1e-3, keep_model=False):
    key = f"{name}__f{f}__s{seed}"; pf = PRED_DIR / f"{key}.npz"
    if pf.exists() and not keep_model:
        z = np.load(pf); return z["pred"].astype(np.float32), float(z["val"])
    set_seed(seed); model = build()
    tr, va, te = D[f]["train"], D[f]["val"], D[f]["test"]; rng = np.random.default_rng(seed)
    model, hist = fit(model, lambda ep: iterate_snap(*tr[:3], bs, True, rng), lambda: iterate_snap(*va[:3], bs, False),
                      loss_snap, CKPT_DIR / f"{key}.pt", EPOCHS, lr, patience=PATIENCE, verbose=False)
    pred = predict(model, iterate_snap(*te[:3], bs, False))
    np.savez_compressed(pf, pred=pred.astype(np.float16), val=hist.val.min())
    return (pred, hist.val.min(), model) if keep_model else (pred, hist.val.min())

for f in FOLD_IDS:
    t0 = time.time()
    PRED[f]["iTransformer"] = [run_snap("iTransformer", lambda: ITransformer(), f, s)[0] for s in SEEDS]
    print(f"iTransformer fold {f}: {time.time()-t0:.1f}s")

## 7. Factorial experiment: graph construction x architecture
10 graphs x 3 architectures x seeds x folds. Response = mean test sMAPE over the reported horizons (and
CC-sMAPE). The graph for the main DGTT model is then chosen by **validation loss**, never test error.

In [ ]:
# ==== 7. FACTORIAL RUNS (resumable; each run checkpointed) ==================
ARCHS = {"GCN-GRU": "gcn", "GAT-Trans": "gat", "DGTT": "dgtt"}

def make_graph_model(arch, g, f, **kw):
    if g == "G8": return GraphForecaster(arch, None, learned=True, use_prior=False, **kw)
    if g == "G0":
        if arch == "dgtt": return GraphForecaster(arch, None, message_passing=False, **kw)
        return GraphForecaster(arch, None, **kw)
    return GraphForecaster(arch, GRAPHS[f][g], **kw)

def summarise(pred, f):
    te = D[f]["test"]; s, _ = evaluate(pred, te[1], te[2], te[3], prep, FOLDS[f])
    return s.sMAPE.mean(), s.CC_sMAPE.mean(), s.MASE.mean()

reg = pd.read_csv(REG_FILE) if REG_FILE.exists() else pd.DataFrame()
done = set(reg.key) if len(reg) else set()
t_all = time.time()
for f in FOLD_IDS:
    for arch_name, arch in ARCHS.items():
        for g in GRAPH_NAMES:
            for s in FACT_SEEDS:
                key = f"FACT_{arch_name}_{g}__f{f}__s{s}"
                if key in done: continue
                t0 = time.time()
                pred, val = run_snap(f"FACT_{arch_name}_{g}", lambda: make_graph_model(arch, g, f), f, s)
                smape, cc, mase = summarise(pred, f)
                row = dict(key=key, arch=arch_name, graph=g, fold=f, seed=s, val_loss=val, sMAPE=smape, CC_sMAPE=cc,
                           MASE=mase, sec=time.time() - t0)
                reg = pd.concat([reg, pd.DataFrame([row])], ignore_index=True); reg.to_csv(REG_FILE, index=False)
                print(f"{key}: sMAPE {smape:.3f} CC {cc:.3f} ({row['sec']:.0f}s)")
print(f"factorial finished in {(time.time()-t_all)/60:.1f} min, {len(reg)} runs")

In [ ]:
# ==== 7.1 VARIANCE DECOMPOSITION (two-way ANOVA with fold blocks, eta-squared, bootstrap CI) ====
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

def eta_sq(df, y):
    """Two-way ANOVA (graph, arch, interaction) with fold as a blocking factor.
    Fold-to-fold level differences are a nuisance, so effect sizes are reported
    (i) as eta-squared of the variance left after removing fold effects and
    (ii) as partial eta-squared."""
    d = df.rename(columns={y: "y"})
    terms = "C(graph) + C(arch) + C(graph):C(arch)" + (" + C(fold)" if d.fold.nunique() > 1 else "")
    a = anova_lm(smf.ols(f"y ~ {terms}", data=d).fit(), typ=2)
    ss_fold = a.loc["C(fold)", "sum_sq"] if "C(fold)" in a.index else 0.0
    a["eta_sq_within"] = a["sum_sq"] / (a["sum_sq"].sum() - ss_fold)
    a["partial_eta_sq"] = a["sum_sq"] / (a["sum_sq"] + a.loc["Residual", "sum_sq"])
    return a

REG = pd.read_csv(REG_FILE)
REG = REG[REG.fold.isin(FOLD_IDS) & REG.seed.isin(FACT_SEEDS)]
rows = []
for y in ("sMAPE", "CC_sMAPE"):
    a = eta_sq(REG, y)
    rng = np.random.default_rng(0); boots = []
    for _ in range(200 if not SMOKE else 20):
        bs = REG.groupby(["graph", "arch"], group_keys=False).apply(
            lambda g: g.sample(len(g), replace=True, random_state=int(rng.integers(1e9))))
        try:
            b = eta_sq(bs, y); boots.append(b.loc["C(graph)", "eta_sq_within"] - b.loc["C(arch)", "eta_sq_within"])
        except Exception:
            pass
    lo, hi = (np.quantile(boots, [0.025, 0.975]) if boots else (np.nan, np.nan))
    for term in a.index:
        if term == "C(fold)": continue
        rows.append(dict(metric=y, term=term.replace("C(", "").replace(")", ""), df=a.loc[term, "df"],
                         F=a.loc[term, "F"], p=a.loc[term, "PR(>F)"], eta_sq=a.loc[term, "eta_sq_within"],
                         partial_eta_sq=a.loc[term, "partial_eta_sq"] if term != "Residual" else np.nan,
                         CI_lo=np.nan, CI_hi=np.nan))
    rows.append(dict(metric=y, term="graph minus arch", df=np.nan, F=np.nan, p=np.nan,
                     eta_sq=a.loc["C(graph)", "eta_sq_within"] - a.loc["C(arch)", "eta_sq_within"],
                     partial_eta_sq=np.nan, CI_lo=lo, CI_hi=hi))
save_table(pd.DataFrame(rows), "tab_factorial_anova",
           "Variance decomposition of test accuracy across the graph x architecture factorial (fold as block). "
           "eta_sq = share of within-fold variance; 'graph minus arch' gives the difference in eta-squared with a "
           "bootstrap 95% CI (resampling runs within cells).", nd=4)

fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.5), sharey=False)
for j, y in enumerate(("sMAPE", "CC_sMAPE")):
    R_ = REG.assign(dev=REG[y] - REG.groupby("fold")[y].transform("mean"))       # remove fold (block) effect
    m = R_.groupby(["graph", "arch"])["dev"].agg(["mean", "std"]).reset_index()
    for c, arch_name in enumerate(ARCHS):
        s = m[m.arch == arch_name].set_index("graph").reindex(GRAPH_NAMES)
        ax[j].errorbar(range(len(GRAPH_NAMES)), s["mean"], yerr=s["std"], marker="o", ms=4, lw=1.4, capsize=2,
                       color=PALETTE[c], label=arch_name)
    ax[j].set_xticks(range(len(GRAPH_NAMES)), GRAPH_NAMES, rotation=45); ax[j].set_ylabel("deviation from fold mean (pp)")
    ax[j].set_title(f"({'ab'[j]}) {y.replace('_', '-')} by graph and architecture")
ax[1].legend(loc="center left", bbox_to_anchor=(1.0, 0.5)); fig.tight_layout(); savefig(fig, "fig_factorial_interaction")

## 8. Main DGTT model, ablations and the main comparison
The prior graph of the full DGTT is the construction with the lowest mean **validation** loss in the factorial.

In [ ]:
# ==== 8. SELECT PRIOR BY VALIDATION + ABLATIONS =============================
vsel = REG[(REG.arch == "DGTT") & ~REG.graph.isin(["G0", "G8"])].groupby("graph").val_loss.mean()
BEST_G = vsel.idxmin(); print("DGTT prior selected on validation:", BEST_G, vsel.round(4).to_dict())
ABL = {"DGTT (full)": {},
       "- dynamic q.k term": dict(dynamic=False),
       "- learned static embeddings": dict(static=False),
       "- statistical prior": dict(use_prior=False),
       "- graph attention (mean agg.)": dict(graph_attn=False),
       "- cross-product message passing": dict(message_passing=False),
       "- temporal attention (MLP enc.)": dict(temporal="mlp"),
       "- product embeddings": dict(prod_emb=False)}
MODELS_DGTT = {}
for f in FOLD_IDS:
    for name, kw in ABL.items():
        t0 = time.time()
        PRED[f][name] = [run_snap(f"ABL_{name}", lambda: GraphForecaster("dgtt", GRAPHS[f][BEST_G], **kw), f, s)[0] for s in SEEDS]
        print(f"{name} fold {f}: {time.time()-t0:.0f}s")
    for gname, arch in (("GCN-GRU", "gcn"), ("GAT-Trans", "gat")):
        PRED[f][f"{gname} ({BEST_G})"] = [run_snap(f"MAIN_{gname}", lambda: make_graph_model(arch, BEST_G, f), f, s)[0] for s in SEEDS]

In [ ]:
# ==== 8.1 EVALUATION OF EVERY MODEL (mean +- sd over seeds, averaged over folds) ====
def eval_model(name):
    per_seed = []
    for s in range(len(PRED[FOLD_IDS[0]][name])):
        fs = []
        for f in FOLD_IDS:
            te = D[f]["test"]; sm, _ = evaluate(PRED[f][name][s], te[1], te[2], te[3], prep, FOLDS[f])
            sm["AUPRC_chg"] = [change_auprc(np.abs(PRED[f][name][s]), te[1], te[2], h) for h in sm.h]
            fs.append(sm)
        per_seed.append(pd.concat(fs).groupby("h").mean(numeric_only=True))
    st = pd.concat(per_seed, keys=range(len(per_seed)))
    return st.groupby(level="h").mean(), st.groupby(level="h").std()

BASE_ORDER = ["Naive", "SNaive-7", "MA-7", "AutoETS", "AutoARIMA", "LightGBM", "LSTM", "PatchTST", "TFT-lite",
              "iTransformer", f"GCN-GRU ({BEST_G})", f"GAT-Trans ({BEST_G})", "DGTT (full)"]
EV = {m: eval_model(m) for m in BASE_ORDER + [a for a in ABL if a != "DGTT (full)"]}
rows = []
for m in BASE_ORDER:
    mu, sd = EV[m]
    r = dict(model=m)
    for h in H_EVAL: r[f"sMAPE@{h}"] = mu.loc[h, "sMAPE"]
    r.update(sMAPE=mu.sMAPE.mean(), sMAPE_sd=sd.sMAPE.mean() if len(sd.dropna()) else np.nan,
             CC_sMAPE=mu.CC_sMAPE.mean(), WAPE=mu.WAPE.mean(), MASE=mu.MASE.mean(), MAE=mu.MAE.mean(),
             RMSE=mu.RMSE.mean(), AUPRC_chg7=mu.loc[7, "AUPRC_chg"] if 7 in mu.index else np.nan)
    rows.append(r)
MAIN = save_table(pd.DataFrame(rows), "tab_main_results",
                  "Test accuracy on active food SKUs, averaged over three rolling-origin folds (mean over seeds; "
                  "sMAPE_sd = s.d. across seeds). Horizon-specific columns report sMAPE (%). AUPRC_chg7 = AUPRC "
                  "for detecting a price change within 7 days (using |forecast| as the score).",
                  nd=3, bold_min=[f"sMAPE@{h}" for h in H_EVAL] + ["sMAPE", "CC_sMAPE", "WAPE", "MASE", "MAE", "RMSE"],
                  bold_max=["AUPRC_chg7"])
rows = []
full = EV["DGTT (full)"][0]
for m in ABL:
    mu, sd = EV[m]
    rows.append(dict(variant=m, sMAPE=mu.sMAPE.mean(), sd=sd.sMAPE.mean(), CC_sMAPE=mu.CC_sMAPE.mean(),
                     delta_sMAPE=mu.sMAPE.mean() - full.sMAPE.mean(), delta_CC=mu.CC_sMAPE.mean() - full.CC_sMAPE.mean()))
ABLT = save_table(pd.DataFrame(rows), "tab_ablation",
                  f"DGTT ablations (prior graph {BEST_G}). Delta = variant minus full model; positive = component helps.",
                  nd=3, bold_min=["sMAPE", "CC_sMAPE"])

In [ ]:
# ==== 8.2 FIGURES: horizon curves + ablation ================================
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.6))
show = ["Naive", "AutoARIMA", "LightGBM", "PatchTST", "iTransformer", f"GAT-Trans ({BEST_G})", "DGTT (full)"]
for c, m in enumerate(show):
    mu = EV[m][0]
    ax[0].plot(mu.index, mu.sMAPE, marker="o", color=PALETTE[c], label=m)
    ax[1].plot(mu.index, mu.CC_sMAPE, marker="o", color=PALETTE[c], label=m)
for a, t in zip(ax, ("(a) sMAPE (all targets)", "(b) CC-sMAPE (targets with a price change)")):
    a.set_xscale("log"); a.set_xticks(H_EVAL, H_EVAL); a.minorticks_off(); a.set_xlabel("Horizon (days)"); a.set_ylabel("%"); a.set_title(t)
ax[1].legend(loc="center left", bbox_to_anchor=(1.0, 0.5)); fig.tight_layout(); savefig(fig, "fig_horizon_curves")

fig, ax = plt.subplots(figsize=(COL1, 2.4))
a = ABLT[ABLT.variant != "DGTT (full)"].sort_values("delta_CC")
ax.barh(a.variant, a.delta_CC, color=[PALETTE[0] if v >= 0 else PALETTE[1] for v in a.delta_CC])
ax.axvline(0, color=GREY, lw=0.8); ax.set_xlabel("Change in CC-sMAPE vs full model (pp)")
ax.set_title("Ablation: removing each component"); fig.tight_layout(); savefig(fig, "fig_ablation")

## 9. Statistical significance
DGTT against every baseline: Diebold-Mariano (HLN correction) on per-origin cross-sectional sAPE,
Wilcoxon signed-rank on per-product sAPE, and a bootstrap 95% CI of the per-product sMAPE difference;
p-values are Holm-corrected within each horizon.

In [ ]:
# ==== 9. SIGNIFICANCE TESTS =================================================
def pooled(name, h):
    """seed-averaged prediction -> per-origin loss series and per-product sAPE, pooled over folds."""
    lo, pp = [], []
    for f in FOLD_IDS:
        p = np.mean(PRED[f][name], 0); te = D[f]["test"]
        lo.append(per_origin_loss(p, te[1], te[2], te[3], prep, h))
        _, per = evaluate(p, te[1], te[2], te[3], prep, FOLDS[f], horizons=[h]); pp.append(per[h])
    return pd.concat(lo, ignore_index=True), pd.concat(pp).groupby(level=0).mean()

rows = []
for h in H_EVAL:
    lp, pp = pooled("DGTT (full)", h); block = []
    for m in BASE_ORDER[:-1]:
        lb, pb = pooled(m, h)
        dm, p_dm = dm_test(lp.values, lb.values, h)
        p_w = wilcoxon_paired(pp, pb)
        j = pp.index.intersection(pb.index); mean, lo_, hi_ = bootstrap_ci((pb[j] - pp[j]).values)
        block.append(dict(h=h, baseline=m, DM=dm, p_DM=p_dm, p_Wilcoxon=p_w, dsMAPE=mean, CI_lo=lo_, CI_hi=hi_))
    b = pd.DataFrame(block); b["p_DM_holm"] = holm(b.p_DM); b["p_W_holm"] = holm(b.p_Wilcoxon); rows.append(b)
SIG = save_table(pd.concat(rows, ignore_index=True), "tab_significance",
                 "DGTT vs baselines. DM < 0 means DGTT is more accurate; dsMAPE = baseline minus DGTT per-product "
                 "sMAPE (positive favours DGTT) with bootstrap 95% CI; Holm-adjusted p-values.", nd=4)

## 10. Heterogeneity: volatility, activity tier, category, correlation strength

In [ ]:
# ==== 10. HETEROGENEITY =====================================================
def per_product_smape(name):
    out = []
    for f in FOLD_IDS:
        p = np.mean(PRED[f][name], 0); te = D[f]["test"]
        _, per = evaluate(p, te[1], te[2], te[3], prep, FOLDS[f])
        out.append(pd.concat(per, axis=1).mean(1))
    return pd.concat(out).groupby(level=0).mean()

HET_MODELS = ["Naive", "LightGBM", "PatchTST", "iTransformer", "DGTT (full)"]
PP = pd.DataFrame({m: per_product_smape(m) for m in HET_MODELS})
info = meta.loc[PP.index, ["tier", "vol_bucket", "category"]].copy()
C4 = GRAPHS[FOLD_IDS[-1]]["_corr"].copy(); np.fill_diagonal(C4, np.nan)
pos = pd.Series(np.arange(N), index=IDX)
strength = pd.Series(np.nanmax(np.abs(C4), 1), index=IDX).loc[PP.index]
info["corr_strength"] = pd.qcut(strength.rank(method="first"), 3, labels=["weak", "medium", "strong"])
rows = []
for col in ("vol_bucket", "tier", "corr_strength"):
    for lev, g in info.groupby(col, observed=True):
        r = dict(split=col, group=str(lev), n=len(g))
        for m in HET_MODELS: r[m] = PP.loc[g.index, m].mean()
        rows.append(r)
save_table(pd.DataFrame(rows), "tab_heterogeneity", "Test sMAPE (%) by product subgroup.", nd=3)

rel = (1 - PP[HET_MODELS[1:]].div(PP["Naive"].replace(0, np.nan), axis=0)) * 100
rc = rel.join(info.category).groupby("category").mean()
rc = rc.loc[info.category.value_counts().index[: 15].intersection(rc.index)]
fig, ax = plt.subplots(figsize=(COL1 + 0.6, 3.4))
vmax = np.nanmax(np.abs(rc.values)) or 1
im = ax.imshow(rc.values, cmap="RdBu", vmin=-vmax, vmax=vmax, aspect="auto"); ax.grid(False)
ax.set_xticks(range(rc.shape[1]), rc.columns, rotation=45, ha="right"); ax.set_yticks(range(len(rc)), rc.index, fontsize=6)
for i in range(rc.shape[0]):
    for j in range(rc.shape[1]):
        ax.text(j, i, f"{rc.values[i,j]:.0f}", ha="center", va="center", fontsize=5.5)
fig.colorbar(im, ax=ax, label="sMAPE improvement over Naive (%)", shrink=0.8)
ax.set_title("Relative improvement over Naive by category"); fig.tight_layout(); savefig(fig, "fig_category_heatmap")

## 11. Interpretability: what does DGTT's graph learn, and how dynamic is it?

In [ ]:
# ==== 11. LEARNED DYNAMIC GRAPH =============================================
f = FOLD_IDS[-1]
_, _, model = run_snap("ABL_DGTT (full)", lambda: GraphForecaster("dgtt", GRAPHS[f][BEST_G]), f, SEEDS[0], keep_model=True)
te = D[f]["test"]; ATT = []
with torch.no_grad():
    for b in iterate_snap(*te[:3], 2, False):
        model(b["x"]); ATT.append(model.layers[-1].last_attn.float().cpu().numpy())
ATT = np.concatenate(ATT)                                              # (k, N, N)
topk_sets = [(a > 0) for a in ATT]
jac = [((topk_sets[t] & topk_sets[t - 1]).sum() / max((topk_sets[t] | topk_sets[t - 1]).sum(), 1)) for t in range(1, len(ATT))]
O_te = te[3][:, 0, 1]
fig, ax = plt.subplots(figsize=(COL1, 2.2))
ax.plot(prep["dates"][O_te[1:]], jac, color=PALETTE[0], label="edge-set Jaccard (t, t-1)")
for d in O_te[1:][prep["batch_day"][O_te[1:]]]: ax.axvline(prep["dates"][d], color=PALETTE[1], lw=0.8)
for d in O_te[1:][prep["cf_day"][O_te[1:]]]: ax.axvline(prep["dates"][d], color=GREY, lw=0.5, ls=":")
ax.set_ylabel("Jaccard"); ax.set_title("Stability of the learned graph (dotted = outage day)")
ax.tick_params(axis="x", rotation=30); fig.tight_layout(); savefig(fig, "fig_dynamic_graph_stability")

Am = ATT.mean(0); learned = Am > np.quantile(Am, 1 - K_GRAPH / N)
rows = []
for g in GRAPH_NAMES[1:-1]:
    E = GRAPHS[f][g] > 0
    rows.append(dict(graph=g, overlap_with_learned=(E & learned).sum() / max(learned.sum(), 1)))
e = np.nonzero(learned)
rows.append(dict(graph="learned: same-category share", overlap_with_learned=(CAT[e[0]] == CAT[e[1]]).mean()))
save_table(pd.DataFrame(rows), "tab_learned_graph_overlap",
           "Share of DGTT's (time-averaged) learned edges that also appear in each constructed graph.", nd=3)
active = meta.loc[IDX, "n_changes_train"].to_numpy().argsort()[::-1][:5]
rows = []
for i in active:
    nb = Am[i].argsort()[::-1][:5]
    rows.append(dict(product=prep["names"][IDX[i]][:45], top_neighbours="; ".join(prep["names"][IDX[j]][:28] for j in nb)))
save_table(pd.DataFrame(rows), "tab_learned_neighbours", "Most-attended neighbours of the five most active products.")

## 12. Example forecasts

In [ ]:
# ==== 12. EXAMPLE FORECASTS =================================================
f = FOLD_IDS[-1]; te = D[f]["test"]; o0 = 0
t2 = [i for i in np.argsort(-meta.loc[IDX, "n_changes_train"].to_numpy())[:4]]
fig, axs = plt.subplots(2, 2, figsize=(COL2, 3.6)); axs = axs.ravel()
for a, i in zip(axs, t2):
    o = te[3][o0, i, 1]; P0 = prep["P"][IDX[i], o]
    hist_t = np.arange(o - 59, o + 1); fut_t = np.arange(o + 1, min(o + 31, prep["P"].shape[1]))
    a.plot(prep["dates"][hist_t], prep["P"][IDX[i], hist_t], color="#52514e", lw=1.2, label="history")
    a.plot(prep["dates"][fut_t], prep["P"][IDX[i], fut_t], color="#0b0b0b", lw=1.2, ls="--", label="actual")
    for c, m in enumerate(["Naive", "PatchTST", "DGTT (full)"]):
        p = np.mean(PRED[f][m], 0)[o0, i, : len(fut_t)]
        a.plot(prep["dates"][fut_t], P0 * np.exp(p / 100), color=PALETTE[c], lw=1.4, label=m)
    a.set_title(prep["names"][IDX[i]][:40], fontsize=7); a.tick_params(axis="x", rotation=30, labelsize=6)
    a.set_ylabel("Price (BDT)")
axs[0].legend(fontsize=6); fig.tight_layout(); savefig(fig, "fig_example_forecasts")

## 13. Paper assets
All figures and tables are on Drive; the cell below lists them and writes a short summary of the headline
numbers for the manuscript.

In [ ]:
# ==== 13. ASSET INDEX + RESULTS SUMMARY =====================================
best = MAIN.loc[MAIN.sMAPE.idxmin(), "model"]; best_cc = MAIN.loc[MAIN.CC_sMAPE.idxmin(), "model"]
fa = pd.read_csv(TAB_DIR / "tab_factorial_anova.csv")
eg = fa[(fa.metric == "sMAPE") & (fa.term == "graph")].eta_sq.iloc[0]; ea = fa[(fa.metric == "sMAPE") & (fa.term == "arch")].eta_sq.iloc[0]
summary = f"""# Paper 1 - results summary ({time.strftime('%Y-%m-%d')})
* Products: {N} active food SKUs; folds {FOLD_IDS}; seeds {SEEDS}; factorial seeds {FACT_SEEDS}.
* Best model by sMAPE: {best}; by CC-sMAPE: {best_cc}.
* DGTT prior graph chosen on validation: {BEST_G}.
* Factorial (sMAPE): eta2(graph) = {eg:.3f}, eta2(architecture) = {ea:.3f} -> the claim "graph construction
  matters more than architecture" is {'SUPPORTED' if eg > ea else 'NOT supported'} on this data (see CI in tab_factorial_anova).
"""
(PAPER_DIR / "RESULTS_SUMMARY.md").write_text(summary); print(summary)
print("figures:", sorted(p.name for p in FIG_DIR.glob("*.pdf")))
print("tables :", sorted(p.name for p in TAB_DIR.glob("*.tex")))